# Recapitulation of published clustering and labels — Ye 2026 sepsis and Stephenson 2025 ESKD COVID

Validation notebook for the two re-annotated public datasets. Outputs were executed in the cloud workspace on the
same data that produced the `.h5ad` files on your drive:

| file | content |
|---|---|
| `sepsis_GSE279452_full/sepsis_GSE279452_recap.h5ad` | 855,536 QC-passed singlets × 37,4xx genes (raw counts in `X`), `obs['celltype_ye2026']` (paper subset names, Fig. 1c), `obs['celltype_major_ye2026']`, `qc_flag`, `in_paper_scheme`, UMAPs and Harmony PCs in `obsm` |
| `covid_eskd/eskd_covid19_recap.h5ad` | deposited object (unchanged) + `cell_type_detailed_recap`, `cell_type_IFNstim_recap` (incl. new `CD14mono_dex`), `recap_subcluster`, `dex_mono_recap`, per-compartment UMAPs |

Methods are stored in `adata.uns['recap_method']` of each file.

In [ ]:
import sys, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, '/home/claude/work'); import store
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 80)
S = '/home/claude/work/sepsis'; Cv = '/home/claude/work/covid'
so = pd.concat([pd.read_csv(f'/mnt/user-data/outputs/sepsis_ann/sepsis_obs_part{i}.csv.gz', keep_default_na=False) for i in range(3)], ignore_index=True)
print(so.shape); so.celltype_major_ye2026.value_counts()

## 1. Sepsis — composition versus the paper (Fig. 1d,e)
Percent of cells in the paper's categories, cells outside the paper scheme (RBC, neutrophils, platelet complexes, low-quality) excluded.

In [ ]:
comp = pd.read_csv('/mnt/user-data/outputs/sepsis_ann/composition_vs_paper.csv'); display(comp)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for a, (coh, d) in zip(ax, comp.groupby('cohort')):
    a.scatter(d.paper_pct, d.recap_pct, s=30, color='#3b6ea5')
    for _, r in d.iterrows(): a.annotate(r.category.replace(' cells', ''), (r.paper_pct, r.recap_pct), fontsize=7)
    m = max(d.paper_pct.max(), d.recap_pct.max()) * 1.05; a.plot([0, m], [0, m], ls='--', c='grey', lw=0.8)
    a.set(title=f'{coh}: r = {np.corrcoef(d.paper_pct, d.recap_pct)[0,1]:.3f}', xlabel='paper %', ylabel='recap %')
plt.tight_layout()

## 2. Sepsis — the 44 subsets plus HSC / megakaryocyte / ILC, by clinical group

In [ ]:
ct = pd.crosstab(so.celltype_ye2026, [so.age_group, so.group]); ct['total'] = ct.sum(1); ct.sort_values('total', ascending=False)

### Rare / disease-specific subsets highlighted in the paper

In [ ]:
rare = ['Mono_c06-CD14-MKI67','B_c08-MKI67','T_CD4_c07-MKI67','T_CD8_c06-MKI67','NK_c04-MKI67','T_CD4_c02-GPR183-NR4A2',
        'T_CD8_c05-GNLY-CCL4hi','NK_c03-PRF1-CCL4hi','NKT_c05-IFNG-CCL4hi','Macrophage','pDC','cDC','gdT','HSC','Megakaryocyte','ILC']
base_ped = (so.age_group == 'paediatric').mean(); base_sep = (so.group == 'sepsis').mean()
t = pd.DataFrame({c: {'cells': int((so.celltype_ye2026 == c).sum()),
                      'paediatric_frac': round((so.age_group[so.celltype_ye2026 == c] == 'paediatric').mean(), 3),
                      'sepsis_frac': round((so.group[so.celltype_ye2026 == c] == 'sepsis').mean(), 3)} for c in rare}).T
print(f'baseline paediatric {base_ped:.3f}, sepsis {base_sep:.3f}'); t

## 3. Sepsis — defining genes per subset (mean log-normalised expression)

In [ ]:
st = store.Store(f'{S}/store'); g = st.var.gene_symbol.astype(str).to_numpy()
keep = so.in_paper_scheme.astype(str).eq('True').to_numpy()
groups, mean, pct, n = store.group_stats(st, np.where(keep)[0], so.celltype_ye2026.to_numpy()[keep])
genes = ['CD3D','CD4','CD8B','LEF1','CCR7','GPR183','NR4A2','AREG','ANXA2','GZMK','GNLY','CCL4','IFNG','FOXP3','MKI67','TRAV1-2','TRDC',
         'KLRF1','FCGR3A','CX3CR1','CXCR4','HLA-DRA','XCL1','TCL1A','CD83','TNF','TLN1','AIM2','CD27','MZB1','CD14','IL1B','S100A8','RETN',
         'VCAN','IFITM1','C1QA','CD1C','LILRA4','CD34','PPBP','KIT']
gi = [int(np.where(g == x)[0][0]) for x in genes]
M = pd.DataFrame(mean[:, gi], index=groups, columns=genes)
order = sorted(groups, key=lambda x: (x.split('_')[0], x))
fig, ax = plt.subplots(figsize=(14, 12)); im = ax.imshow((M.loc[order] / M.max()).to_numpy(), aspect='auto', cmap='Reds')
ax.set_xticks(range(len(genes)), genes, rotation=90, fontsize=8); ax.set_yticks(range(len(order)), order, fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.02, label='scaled mean'); plt.tight_layout()

## 4. Sepsis — UMAPs (Harmony-corrected top-10 PCs, as the paper's top-10 MNN components)

In [ ]:
def scat(ax, U, lab, title, s=0.05):
    cats = pd.Categorical(lab); cmap = plt.get_cmap('tab20', max(20, len(cats.categories)))
    idx = np.random.default_rng(0).permutation(len(U))
    ax.scatter(U[idx, 0], U[idx, 1], c=cats.codes[idx], cmap=cmap, s=s, rasterized=True, linewidths=0)
    for i, c in enumerate(cats.categories):
        m = cats.codes == i
        if m.sum() > 50: ax.text(*np.median(U[m], 0), c, fontsize=6, ha='center')
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
fig, ax = plt.subplots(figsize=(9, 8))
scat(ax, np.load(f'{S}/run_major/umap.npy'), so.celltype_major_ye2026.to_numpy(), 'all cells — paper major categories')

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(16, 15))
for ax, run in zip(axs.ravel(), ['b', 'my', 'cd4', 'cyto']):
    cells = np.load(f'{S}/run_{run}/embed.npz')['cells']; U = np.load(f'{S}/run_{run}/umap.npy')
    scat(ax, U, so.celltype_ye2026.to_numpy()[cells], {'b':'B / plasma','my':'myeloid','cd4':'CD4 T','cyto':'CD8 / NK / NKT / gdT'}[run], s=0.2)
plt.tight_layout()

## 5. COVID ESKD — agreement of re-derived labels with the deposited annotation

In [ ]:
ost = store.Store(f'{Cv}/store'); co = ost.obs
L = pd.read_csv('/mnt/user-data/outputs/covid_ann/covid_obs_recap.csv.gz', index_col=0, keep_default_na=False)
assert (L.index == co.index).all()
for a_, r_ in [('cell_type_detailed', 'cell_type_detailed_recap'), ('cell_type_IFNstim', 'cell_type_IFNstim_recap')]:
    print(r_, 'overall agreement', round((co[a_].astype(str).values == L[r_].values).mean(), 3))
au = co.cell_type_detailed.astype(str).values; rc = L.cell_type_detailed_recap.values
pd.DataFrame({c: {'deposited_n': int((au == c).sum()), 'recovered_frac': round((rc[au == c] == c).mean(), 3), 'recap_n': int((rc == c).sum())}
              for c in sorted(set(au))}).T.sort_values('deposited_n')

In [ ]:
cm = pd.crosstab(pd.Series(au, name='deposited'), pd.Series(rc, name='recap'), normalize='index')
fig, ax = plt.subplots(figsize=(12, 10)); ax.imshow(cm.to_numpy(), cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(cm.shape[1]), cm.columns, rotation=90, fontsize=7); ax.set_yticks(range(cm.shape[0]), cm.index, fontsize=7)
ax.set_xlabel('recap'); ax.set_ylabel('deposited'); plt.tight_layout()

## 6. COVID — dexamethasone-associated monocytes (paper Fig. 5)

In [ ]:
import anndata as ad
mono = ad.read_h5ad(f'{Cv}/mono.h5ad'); lab = L.cell_type_IFNstim_recap.values[mono.obs.cell_index.values]
gg = ost.var.gene.to_numpy()
groups, mean, pct, n = store.group_stats(ost, mono.obs.cell_index.values, lab)
dex = ['CD163','ADAMTS2','SLC1A3','MRC1','FPR1','MTSS1','SESN1','MFGE8','FKBP5','TSC22D3','JUN','CXCL8','IL1B','HLA-DRA','HLA-DRB5']
display(pd.DataFrame(mean[:, [int(np.where(gg == x)[0][0]) for x in dex]], index=groups, columns=dex).round(2))
o = mono.obs.assign(lab=lab, cohort=np.where(mono.obs.sample_date.astype(str).str.endswith('2021'), '2021', '2020'))
display(pd.crosstab(o.lab, [o.cohort, o.WHO_severity_simple]).loc[['CD14mono_dex', 'CD14mono']])
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
scat(ax[0], mono.obsm['X_umap'], lab, 'monocyte re-clustering: recap labels', s=0.3)
ax[1].scatter(*mono.obsm['X_umap'].T, s=0.3, c='lightgrey', rasterized=True); m = lab == 'CD14mono_dex'
ax[1].scatter(*mono.obsm['X_umap'][m].T, s=0.6, c='crimson', rasterized=True); ax[1].set_title('CD14mono_dex'); ax[1].set_xticks([]); ax[1].set_yticks([])

## 7. Loading the delivered files with Scanpy

In [ ]:
# On your machine (paths on F:):
# import scanpy as sc
# sep = sc.read_h5ad(r'F:/PBISC/bulkonly_lora5_20261006/analysis/original_scRNA/sepsis_GSE279452_full/sepsis_GSE279452_recap.h5ad')
# sep = sep[sep.obs.in_paper_scheme].copy()                  # drop RBC / neutrophil / platelet complexes / low-quality, as the paper
# sc.pp.normalize_total(sep, 1e4); sc.pp.log1p(sep); sc.pl.embedding(sep, 'X_umap', color='celltype_ye2026')
# covid: the deposited object carries a float64 'logcounts' layer (~9 GB in RAM); for label work read obs/obsm only:
# import h5py; from anndata.io import read_elem
# with h5py.File(r'F:/PBISC/bulkonly_lora5_20261006/analysis/original_scRNA/covid_eskd/eskd_covid19_recap.h5ad') as f: obs = read_elem(f['obs'])
print('see markdown above')